# 3차시. 결측치·중복·이상치 탐지 및 처리

## 1. 학습목표
titanic·tips 데이터로 결측치 → 중복 → 이상치 순서로 직접 진단하고 처리합니다.

## 2. 선수지식 — 조건 마스크와 평균·중앙값
tips 데이터를 불러와 조건 마스크로 개수를 세고, 평균과 중앙값을 비교합니다.

In [1]:
import pandas as pd                          # 판다스 불러오기
import numpy as np                           # NaN(np.nan) 사용

tips = pd.read_csv("data/raw/tips.csv")      # tips 데이터 불러오기

mask = tips["tip"] >= 5                      # 조건 → True/False
print("팁 5 이상:", mask.sum(), "건")         # True 개수
tip_mean = tips["tip"].mean()                # 팁 평균
print("팁 평균  :", round(tip_mean, 2))       # 소수 둘째 자리까지
print("팁 중앙값:", tips["tip"].median())

팁 5 이상: 28 건
팁 평균  : 3.0
팁 중앙값: 2.9


## 3. 결측치란? — NaN
None·np.nan이 섞인 Series를 만들어 isna로 결측을 확인하고, NaN끼리 비교하면 어떻게 되는지 봅니다.

In [10]:
scores = pd.Series(
    [90,100,None,85] , 
    index=["a", "b", "c", "d"]
)

# isna
# 집계할 때 None 빠짐

## 4. 결측치 파악 — 개수·비율 리포트
titanic에서 7개 열을 불러와 열별 결측 개수와 비율을 표로 정리합니다.

In [ ]:
titanic_df = pd.read_csv("data/raw/titanic.csv")

# 칼럼별로 결측이 몇개나 있을까?
age_median = titanic_df.isna().sum()

# 비율
embarked_mode = round( titanic_df.isna().mean() * 100 , 2)

# 결측 리포트
report = pd.DataFrame(
    {
        "결측갯수" : age_median ,
        "결측비율" : embarked_mode
    }
)

report

,결측갯수,결측비율
survived,0,0.00
pclass,0,0.00
sex,0,0.00
age,177,19.87
sibsp,0,0.00
parch,0,0.00
fare,0,0.00
embarked,2,0.22
class,0,0.00
who,0,0.00


## 5. 결측치 삭제 — drop · dropna
결측이 많은 deck 열을 삭제하고, dropna 옵션에 따라 남는 행 수가 어떻게 달라지는지 비교합니다.

In [30]:
# titanic_df.drop(columns=["deck"])
# titanic_df.dropna()
titanic_df.dropna(subset=["embarked"])

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


## 6. 결측치 채우기 — fillna (중앙값·최빈값)
age는 중앙값, embarked는 최빈값으로 채우고 결측이 모두 사라졌는지 확인합니다.

In [55]:
df = titanic_df.drop(columns=["deck"])

# age
age_median = df["age"].median()
df["age"] = df["age"].fillna(age_median)

# embarked
embarked_mode = df["embarked"].mode()[0]
df["embarked"] = df["embarked"].fillna(embarked_mode)
df["embarked"].isna().sum()

np.int64(0)

## 7. 열마다 다른 값으로 채우기 — fillna(딕셔너리)
딕셔너리로 열마다 다른 값을 채우고, embarked에 Unknown이 몇 개 들어갔는지 확인합니다.

In [61]:
df = titanic_df.drop(columns=["deck"])
df = df.fillna({"age":300 , "embarked":"Hello"})
df.isna().sum()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    2
alive          0
alone          0
dtype: int64

## 8. 중복 찾기 — duplicated
중복이 섞인 회원 데이터를 만들어, 판단 기준(subset)에 따라 중복 개수가 달라지는 것을 확인합니다.

In [67]:
members = pd.DataFrame({
    "user_id" : [101, 102, 102, 103, 104, 104] ,
    "name" : ["김태희" , "한예슬" , "한예슬" , "전지현" , "카리나" , "카리나"] ,
    "city" : ["서울" , "대전" , "대전", "대구", "부산", "서울"] ,
})

members.duplicated(subset=["user_id"])

0    False
1    False
2     True
3    False
4    False
5     True
dtype: bool

## 9. 중복 제거 — drop_duplicates
중복 행을 제거하고, 같은 user_id는 마지막(최신) 행만 남긴 뒤 인덱스를 다시 매깁니다.

In [76]:
# result = members.drop_duplicates()
result = members.drop_duplicates(subset=["user_id"] , keep="last")
result = result.reset_index(drop=True)
result

,user_id,name,city
0,101,김태희,서울
1,102,한예슬,대전
2,103,전지현,대구
3,104,카리나,서울


## 10. 실제 데이터의 중복 — 지워도 될까?
tips와 titanic의 중복 행을 찾아보고, 정말 지워도 되는 데이터인지 확인합니다.

In [ ]:
tips
titanic = titanic_df

print("tips dup : " , tips.duplicated().sum())
print("titanic dup : " , titanic.duplicated().sum())

tips dup :  1
titanic dup :  107


## 11. IQR로 이상치 경계 정하기
tips의 tip 열로 Q1·Q3·IQR을 구해 이상치 정상 범위를 계산합니다.

In [ ]:
q1 = tips["tip"].quantile(0.25)
q3 = tips["tip"].quantile(0.75)

iqr = q3 - q1

clean_df = iqr * 1.5
lower = q1 - clean_df
upper = q3 + clean_df

print(lower)
print(upper)


-0.34375
5.90625


## 12. 이상치 탐지 — 범위 밖 마스크
정상 범위를 벗어난 행을 마스크로 찾아 이상치 개수와 내용을 확인합니다.

In [92]:
is_outlier = (tips["tip"] > upper) | (tips["tip"] < lower)
tips.loc[is_outlier]

,total_bill,tip,sex,smoker,day,time,size
23,39.42,7.58,Male,No,Sat,Dinner,4
47,32.40,6.00,Male,No,Sun,Dinner,4
59,48.27,6.73,Male,No,Sat,Dinner,4
141,34.30,6.70,Male,No,Thur,Lunch,6
170,50.81,10.00,Male,Yes,Sat,Dinner,3
183,23.17,6.50,Male,Yes,Sun,Dinner,4
212,48.33,9.00,Male,No,Sat,Dinner,4
214,28.17,6.50,Female,Yes,Sat,Dinner,3
239,29.03,5.92,Male,No,Sat,Dinner,3


## 13. 이상치 처리 — 제거 vs clip
이상치를 제거한 결과와 clip으로 자른 결과의 행 수·최댓값을 비교합니다.

In [ ]:
removed = tips[~is_outlier]
removed.shape
removed["tip"].max()

np.float64(5.85)

In [105]:
tips_copy = tips.copy()
tips_copy["tip"] = tips["tip"].clip(lower=lower , upper=upper)
tips_copy

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01000,Female,No,Sun,Dinner,2
1,10.34,1.66000,Male,No,Sun,Dinner,3
2,21.01,3.50000,Male,No,Sun,Dinner,3
3,23.68,3.31000,Male,No,Sun,Dinner,2
4,24.59,3.61000,Female,No,Sun,Dinner,4
...,...,...,...,...,...,...,...
239,29.03,5.90625,Male,No,Sat,Dinner,3
240,27.18,2.00000,Female,Yes,Sat,Dinner,2
241,22.67,2.00000,Male,Yes,Sat,Dinner,2
242,17.82,1.75000,Male,No,Sat,Dinner,2


## 14. 종합 실습 — clean_titanic 함수
결측치·이상치 처리 과정을 함수 하나로 묶고, 처리 전·후 결과를 비교합니다.

In [ ]:
def clean_titanic(titanic_df):
    # 결측치 (deck:제거 ,  나이:중앙값 , 항구:최빈값)
    result = titanic_df.drop(columns=["deck"])
    age_median = result["age"].median()
    result["age"] = result["age"].fillna(age_median)

    # 이상치

    return result


clean_df = clean_titanic(titanic_df)
clean_df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,Cherbourg,yes,True
